# CA1 LFP band power with MNE filters (Dandiset 001695)

Builds band-pass FIR filters with MNE, applies them to the session's single CA1 LFP channel
(`Best_Ripple_channel_LFP_CA1`, about 1250 Hz), takes the Hilbert envelope and bins the band power.
The result is a small table (one row per bin, one column per band) written to storage, which the
Grassmannian notebook (`../Hp_grassmannian_pipeline_001695.ipynb`) reads to colour its windows.
It also pulls the scored sleep states out of the file (pynapple drops those labels) and saves them
next to the band-power table.

* kernel: `hydrangea_lfp_mne_env`; create it on the server with
  `bash conda_and_jupyterkernel_lfp_mne.sh` (this folder)
* the FIR filters are saved beside this notebook in `filters/` (taps + band definitions), so the
  Grassmannian notebook never needs MNE
* outputs: `storage/grassmanian_test/Hp_001695/lfp/bandpower_<session>.npz` and
  `sleepstates_<session>.csv` (derived data that saves re-filtering; figures are only displayed)

In [ ]:
# ============ storage: same roots as the Grassmannian notebook, one folder deeper ============
from pathlib import Path

STORAGE_DIR   = Path("../../../storage").resolve()
DOWNLOAD_DIR  = STORAGE_DIR / "dandi_downloads"                            # INPUTS (NWB files)
DATASET_LABEL = "Hp_001695"
LFP_OUT_DIR   = STORAGE_DIR / "grassmanian_test" / DATASET_LABEL / "lfp"   # band-power tables, sleep states
FILTER_DIR    = Path("filters").resolve()                                   # FIR filters live in the repo, beside this notebook
LFP_OUT_DIR.mkdir(parents=True, exist_ok=True)
FILTER_DIR.mkdir(parents=True, exist_ok=True)
print("inputs :", DOWNLOAD_DIR)
print("filters:", FILTER_DIR)
print("outputs:", LFP_OUT_DIR)

In [ ]:
# ============ config ============
import json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sps
import mne
import pynapple as nap

SESSION_FILE = DOWNLOAD_DIR / "sub-M01" / "sub-M01_ses-20240313T100000_behavior+ecephys.nwb"   # same session as the Grassmannian notebook
LFP_KEY      = "Best_Ripple_channel_LFP_CA1"      # the one LFP channel in the file (pynapple key)
BIN_MS       = 50.0                               # band power is averaged in bins of this length
BANDS = {                                          # Hz, mouse hippocampus
    "delta":      (1.0,   4.0),
    "theta":      (6.0,  12.0),
    "beta":      (15.0,  30.0),
    "slow_gamma": (30.0,  55.0),
    "fast_gamma": (65.0, 100.0),
    "ripple":   (130.0, 250.0),
}
FIR = dict(method="fir", fir_design="firwin", fir_window="hamming", phase="zero",
           l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto")


def session_key_from_path(path):
    sub = re.search(r"sub-([^_/]+)", str(path))
    ses = re.search(r"ses-([^_/]+)", str(path))
    return "%s_%s" % (sub.group(1), ses.group(1)) if (sub and ses) else Path(path).stem


SESSION_KEY = session_key_from_path(SESSION_FILE)
print("session:", SESSION_KEY, "| mne", mne.__version__)

In [ ]:
# ============ load the LFP channel and the epochs ============
t_load = time.time()
data = nap.load_file(str(SESSION_FILE))
lfp = data[LFP_KEY]
t_lfp = np.asarray(lfp.index.values, dtype=float)
x_lfp = np.asarray(lfp.values, dtype=float).reshape(len(t_lfp), -1)[:, 0]
fs = 1.0 / np.median(np.diff(t_lfp[:100000]))
print("LFP: %d samples, %.1f s, fs = %.2f Hz, loaded in %.1fs"
      % (len(x_lfp), t_lfp[-1] - t_lfp[0], fs, time.time() - t_load))

epochs = {}
if "SleepStates" in data.keys():
    ep = data["SleepStates"]
    epochs["sleep"] = np.column_stack([np.asarray(ep.start, float), np.asarray(ep.end, float)])
if "Position" in data.keys():
    pos = data["Position"]
    epochs["maze"] = np.array([[float(pos.index[0]), float(pos.index[-1])]])
print("epochs:", {k: v.shape for k, v in epochs.items()})

fig, ax = plt.subplots(figsize=(14, 2.5))
step = max(1, int(fs // 50))                       # ~50 samples/s for the overview
ax.plot(t_lfp[::step], x_lfp[::step], lw=0.3, color="0.3")
for s, e in epochs.get("maze", []):
    ax.axvspan(s, e, color="tab:green", alpha=0.15, lw=0)
for s, e in epochs.get("sleep", []):
    ax.axvspan(s, e, color="tab:orange", alpha=0.10, lw=0)
ax.set(xlabel="time (s)", ylabel="LFP", title="%s (green = maze, orange = SleepStates intervals)" % LFP_KEY)
plt.tight_layout(); plt.show()

In [ ]:
# ============ scored sleep states: pynapple drops the labels, pynwb keeps them ============
import pynwb

sleep_states = None
try:
    with pynwb.NWBHDF5IO(str(SESSION_FILE), "r", load_namespaces=True) as io:
        nwbfile = io.read()
        if "SleepStates" in nwbfile.intervals:
            ss = nwbfile.intervals["SleepStates"].to_dataframe().reset_index(drop=True)
            label_cols = [c for c in ss.columns if c not in ("start_time", "stop_time", "tags", "timeseries")]
            print("SleepStates columns:", list(ss.columns))
            if label_cols:
                sleep_states = (ss[["start_time", "stop_time", label_cols[0]]]
                                .rename(columns={label_cols[0]: "state"}))
                sleep_states["state"] = sleep_states["state"].astype(str)
                print("using %r as the state label:" % label_cols[0])
                print(sleep_states.groupby("state").apply(
                    lambda g: pd.Series({"intervals": len(g), "seconds": (g.stop_time - g.start_time).sum()})))
                sleep_states.to_csv(LFP_OUT_DIR / ("sleepstates_%s.csv" % SESSION_KEY), index=False)
                print("wrote", LFP_OUT_DIR / ("sleepstates_%s.csv" % SESSION_KEY))
            else:
                print("SleepStates has no label column; nothing written")
        else:
            print("no SleepStates interval table in the file")
except Exception as exc:
    print("could not read SleepStates with pynwb:", repr(exc))

In [ ]:
# ============ design the band-pass FIR filters with MNE; store them beside this notebook ============
filters = {}
for name, (lo, hi) in BANDS.items():
    taps = mne.filter.create_filter(None, fs, l_freq=lo, h_freq=hi, verbose=False, **FIR)
    filters[name] = np.asarray(taps, dtype=float)
    np.savez(FILTER_DIR / ("fir_%s_%.0fHz.npz" % (name, fs)),
             taps=filters[name], l_freq=lo, h_freq=hi, fs=fs, design=json.dumps(FIR))
(FILTER_DIR / "bands.json").write_text(json.dumps({"fs": fs, "bands": BANDS, "fir": FIR, "lfp_key": LFP_KEY}, indent=2))
print({k: "%d taps (%.2f s)" % (len(v), len(v) / fs) for k, v in filters.items()})

fig, ax = plt.subplots(figsize=(10, 4))
for name, taps in filters.items():
    w, h = sps.freqz(taps, worN=2 ** 15, fs=fs)
    ax.plot(w, 20 * np.log10(np.abs(h) + 1e-12), lw=1.2, label="%s %g-%g Hz" % (name, *BANDS[name]))
ax.set(xscale="log", xlim=(0.5, fs / 2), ylim=(-80, 5), xlabel="frequency (Hz)", ylabel="gain (dB)",
       title="FIR band-pass responses (MNE firwin, zero phase)")
ax.legend(fontsize=8, frameon=False); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

In [ ]:
# ============ apply the stored filters, Hilbert envelope, bin the power ============
from scipy.fft import next_fast_len

bin_s = BIN_MS / 1000.0
t0 = t_lfp[0]
bin_idx = np.floor((t_lfp - t0) / bin_s).astype(np.int64)
n_bins = int(bin_idx.max()) + 1
n_per_bin = np.bincount(bin_idx, minlength=n_bins).astype(float)
t_center = t0 + (np.arange(n_bins) + 0.5) * bin_s
nfft = next_fast_len(len(x_lfp))

power = np.zeros((n_bins, len(BANDS)), dtype=np.float32)
for j, (name, taps) in enumerate(filters.items()):
    t_b = time.time()
    xf = sps.oaconvolve(x_lfp, taps, mode="same")                   # zero phase: symmetric taps, centred output
    env2 = np.abs(sps.hilbert(xf, N=nfft)[:len(xf)]) ** 2           # squared Hilbert envelope = instantaneous power
    power[:, j] = np.bincount(bin_idx, weights=env2, minlength=n_bins) / np.maximum(n_per_bin, 1)
    del xf, env2
    print("%-11s done in %5.1fs | median power %.3g" % (name, time.time() - t_b, np.median(power[:, j])))
power[n_per_bin == 0] = np.nan
print("band power table:", power.shape, "(bins x bands) at %g ms" % BIN_MS)

In [ ]:
# ============ save the band-power table (derived data: saves the filtering next time) ============
out = LFP_OUT_DIR / ("bandpower_%s.npz" % SESSION_KEY)
np.savez_compressed(out, t_center=t_center, power=power, bands=np.array(list(BANDS)),
                    band_edges=np.array(list(BANDS.values()), dtype=float), bin_ms=BIN_MS, fs=fs,
                    lfp_key=LFP_KEY, session=SESSION_KEY, n_samples_per_bin=n_per_bin)
out.with_suffix(".json").write_text(json.dumps(
    {"session": SESSION_KEY, "lfp_key": LFP_KEY, "fs": fs, "bin_ms": BIN_MS, "bands": BANDS, "fir": FIR,
     "n_bins": int(n_bins), "source_file": str(SESSION_FILE)}, indent=2))
print("wrote", out)

In [ ]:
# ============ sanity plots: band power over the session, PSD by epoch, theta/delta by epoch ============
logp = np.log10(power + 1e-30)
band_list = list(BANDS)
td = logp[:, band_list.index("theta")] - logp[:, band_list.index("delta")]
sm = max(1, int(round(5.0 / bin_s)))               # 5 s running mean, display only

fig, axes = plt.subplots(len(BANDS) + 1, 1, figsize=(14, 1.6 * (len(BANDS) + 1)), sharex=True)
for j, name in enumerate(band_list):
    y = pd.Series(logp[:, j]).rolling(sm, center=True, min_periods=1).mean().values
    axes[j].plot(t_center, y, lw=0.5, color="0.2")
    axes[j].set_ylabel("log10\n%s" % name, fontsize=8)
axes[-1].plot(t_center, pd.Series(td).rolling(sm, center=True, min_periods=1).mean().values, lw=0.5, color="tab:purple")
axes[-1].set_ylabel("theta/delta\n(log10)", fontsize=8)
axes[-1].set_xlabel("time (s)")
for ax in axes:
    for s, e in epochs.get("maze", []):
        ax.axvspan(s, e, color="tab:green", alpha=0.15, lw=0)
axes[0].set_title("band power: 5 s running mean of log10 Hilbert-envelope power (green = maze)", fontsize=10)
plt.tight_layout(); plt.show()


def psd_in(intervals, max_s=600):
    """Welch PSD of the raw LFP inside the given intervals (first max_s of each)."""
    segs = []
    for s, e in intervals:
        a, b = np.searchsorted(t_lfp, [s, min(e, s + max_s)])
        if b - a > fs * 10:
            segs.append(x_lfp[a:b])
    if not segs:
        return None, None
    return sps.welch(np.concatenate(segs), fs=fs, nperseg=int(4 * fs))


fig, ax = plt.subplots(figsize=(7, 4))
for name, color in (("maze", "tab:green"), ("sleep", "tab:orange")):
    if name in epochs:
        f, p = psd_in(epochs[name])
        if f is not None:
            ax.loglog(f, p, color=color, lw=1.2, label=name)
for name, (lo, hi) in BANDS.items():
    ax.axvspan(lo, hi, color="0.92", lw=0)
ax.set(xlim=(0.5, fs / 2), xlabel="frequency (Hz)", ylabel="PSD", title="LFP power spectrum by epoch (grey = bands)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

lab = np.full(n_bins, "other", dtype=object)
for name, iv in epochs.items():                     # later epochs override earlier ones (maze over sleep)
    for s, e in iv:
        lab[(t_center >= s) & (t_center < e)] = name
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, color in (("sleep", "tab:orange"), ("maze", "tab:green"), ("other", "tab:blue")):
    m = (lab == name) & np.isfinite(td)
    if m.any():
        ax.hist(td[m], bins=80, histtype="step", density=True, color=color, lw=1.3, label="%s (%d bins)" % (name, m.sum()))
ax.set(xlabel="log10 theta/delta power ratio per %g ms bin" % BIN_MS, ylabel="density")
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()